# Day 5: 异常处理

## 学习内容
- try/except/finally
- 自定义异常
- 完善异常处理

## 1. 基础异常处理

In [ ]:
# 示例 1: 除零异常
print("1. 除零异常处理:")
try:
    result = 10 / 0
except ZeroDivisionError:
    print("   ❌ 错误：除数不能为零！")
    result = None
print(f"   结果：{result}")

In [ ]:
# 示例 2: 值错误异常
print("2. 值错误异常处理:")
try:
    age = int("不是数字")
except ValueError:
    print("   ❌ 错误：无法转换为整数！")
    age = 0
print(f"   年龄：{age}")

# 正常转换
age = int("25")
print(f"   正常转换：{age}")

In [ ]:
# 示例 3: 索引错误异常
print("3. 索引错误异常处理:")
try:
    items = [1, 2, 3]
    value = items[10]
except IndexError:
    print("   ❌ 错误：索引超出范围！")
    value = None
print(f"   值：{value}")

## 2. 多个异常处理

In [ ]:
def safe_divide(a, b):
    """安全的除法运算"""
    try:
        result = a / b
        return {"success": True, "result": result, "error": None}
    except ZeroDivisionError:
        return {"success": False, "result": None, "error": "除数不能为零"}
    except TypeError:
        return {"success": False, "result": None, "error": "输入必须是数字"}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}

# 测试不同情况
test_cases = [
    (10, 2),      # 正常
    (10, 0),      # 除零
    ("10", 2),    # 类型错误
    (10, "2"),    # 类型错误
]

for a, b in test_cases:
    print(f"\n计算 {a} / {b}:")
    result = safe_divide(a, b)
    if result["success"]:
        print(f"   ✅ 结果：{result['result']}")
    else:
        print(f"   ❌ 错误：{result['error']}")

## 3. finally 子句

In [ ]:
def demo_finally():
    """finally 子句演示"""
    
    def open_file(filename):
        """模拟文件操作"""
        file = None
        try:
            print(f"\n尝试打开文件：{filename}")
            file = f"FileHandle({filename})"
            print(f"   文件已打开：{file}")

            if "error" in filename:
                raise FileNotFoundError(f"文件 {filename} 不存在")

            content = "文件内容预览..."
            print(f"   读取内容：{content}")
            return content

        except FileNotFoundError as e:
            print(f"   ❌ 错误：{e}")
            return None
        finally:
            if file:
                print(f"   关闭文件：{file}")
            print("   finally: 清理资源完成")

    open_file("data.txt")
    open_file("error_file.txt")

demo_finally()

## 4. 自定义异常

In [ ]:
class InsufficientFundsError(Exception):
    """余额不足异常"""
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"余额不足：当前余额{balance}，需要{amount}")


class InvalidAgeError(Exception):
    """无效年龄异常"""
    def __init__(self, age):
        self.age = age
        super().__init__(f"无效年龄：{age}，必须在 0-150 之间")


class ValidationError(Exception):
    """通用验证异常"""
    def __init__(self, field, message):
        self.field = field
        self.message = message
        super().__init__(f"{field}: {message}")

In [ ]:
class BankAccount:
    """银行账户类（用于演示自定义异常）"""

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        """存款"""
        if amount <= 0:
            raise ValidationError("存款金额", "必须大于 0")
        self.balance += amount
        print(f"✅ 存入 {amount}，当前余额：{self.balance}")
        return self.balance

    def withdraw(self, amount):
        """取款"""
        if amount <= 0:
            raise ValidationError("取款金额", "必须大于 0")
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        print(f"✅ 取出 {amount}，当前余额：{self.balance}")
        return self.balance

In [ ]:
# 测试银行账户
account = BankAccount("张三", 1000)

operations = [
    ("deposit", 500),    # 成功
    ("withdraw", 300),   # 成功
    ("withdraw", 2000),  # 余额不足
    ("deposit", -100),   # 验证错误
]

for op, amount in operations:
    print(f"\n执行操作：{op}({amount})")
    try:
        if op == "deposit":
            account.deposit(amount)
        else:
            account.withdraw(amount)
    except InsufficientFundsError as e:
        print(f"   ❌ {e}")
    except ValidationError as e:
        print(f"   ❌ {e.field} - {e.message}")

print(f"\n最终余额：{account.balance}")

## 5. 获取用户输入（带异常处理）

In [ ]:
def get_user_age():
    """
    获取用户年龄（带完整异常处理）
    
    Returns:
        有效的年龄整数，或 None
    """
    max_attempts = 3

    for attempt in range(1, max_attempts + 1):
        try:
            user_input = input(f"请输入年龄 (第{attempt}次尝试): ")

            if user_input.lower() == 'q':
                print("用户取消输入")
                return None

            age = int(user_input)

            if age < 0 or age > 150:
                raise ValueError(f"年龄必须在 0-150 之间，当前输入：{age}")

            print(f"✅ 有效年龄：{age}")
            return age

        except ValueError as e:
            print(f"❌ 输入无效：{e}")
            if attempt == max_attempts:
                print("已达到最大尝试次数")
                return None

    return None

# 测试（取消注释运行）
# age = get_user_age()
# if age:
#     print(f"用户年龄：{age}")

## 6. 练习

In [ ]:
# 练习：编写一个函数，安全地读取列表元素
# 如果索引无效，返回默认值

def safe_get(lst, index, default=None):
    """
    安全地获取列表元素
    
    Args:
        lst: 列表
        index: 索引
        default: 默认值（索引无效时返回）
    
    Returns:
        列表元素或默认值
    """
    try:
        return lst[index]
    except IndexError:
        print(f"⚠️ 索引 {index} 超出范围，返回默认值")
        return default

# 测试
fruits = ['苹果', '香蕉', '橙子']
print(safe_get(fruits, 0))  # 苹果
print(safe_get(fruits, 5))  # None
print(safe_get(fruits, 5, "未知"))  # 未知
print(safe_get(fruits, -1))  # 橙子

---
✅ Day 5 完成！